In [30]:
import pandas as pd
import warnings 
warnings.filterwarnings("ignore")

# Control Variable(고객데이터) 추가하기
1. 데이터 호출
2. 기본 고객정보 코딩
2. 데이터 결합


### 1. 데이터 호출
* 분석용데이터, 고객정보, 1년간고객구매이력
* 데이터 결합을 위한 INCS_NO 필요   

### 2. 고객 기본정보 코딩
* 남:0, 여:1
* age - 10,20대, 30대, 40대, 50대, 60대이상
* groupby를 이용해 고객별 연평균구매액 코딩
* 고객정보 + 구매이력 결합

### 3. 데이터 결합
* 분석용데이터 + 고객정보 결합
* join 후 누락된 INCS_NO가 많은지 확인 필요
* 누락된 데이터가 많다면 데이터 새로 다운받아야 할 듯 함

### 1. 데이터 호출

In [5]:
df = pd.read_csv('./data/sess_indicate_cutoff7.csv', index_col=0)
print(df.shape)
df.head(2)

(82294, 10)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off
0,0005ad80698368c16258c365c05c76a21732151843,7,1,0,5,7,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-21 10:19:01.273,50.909091,0
3,000A204228DB4108A69885BF2BCA220E1730848747,70,13,1,23,70,51dea90cd3bb8eb26f2580e7891ebb2d38e1d820b8b85a...,2024-11-06 08:39:48.598,33.965517,1


In [6]:
cust = pd.read_csv('./raw/CUST_INFO.csv')
print(cust.shape)
cust.head(2)

(12286714, 3)


,INCS_NO,AGE,SEX_CD
0,f68d5eb51cad02327221e426d7a8878d26c13500b282bd...,60,F
1,30586a6e4e0bb50f62e007dcb09eee969afceb91486fe8...,33,F


In [49]:
cust_sal = pd.read_csv('./raw/CUST_SAL.csv')
cust_sal['STND_YMD'] = pd.to_datetime(cust_sal['STND_YMD'])
print(cust_sal.shape)
cust_sal.head(2)

(10121785, 3)


,INCS_NO,POS_NET_PRD_SAL_AMT,STND_YMD
0,6f9aa6711d3fd1be2b8c07617f2cb182d8e1557dab56c8...,0.0,2024-06-10
1,47cd28290ce9d34e561bf6234a6ce4846930b6f8225493...,0.0,2024-06-10


### 2. 고객 기본정보 코딩

In [51]:
cust_sal['YEAR'] = cust_sal['STND_YMD'].dt.year 
cust_sal_avg = cust_sal.groupby(['INCS_NO', 'YEAR']).POS_NET_PRD_SAL_AMT.mean().reset_index() 
cust_sal_avg = cust_sal_avg.rename(columns={'POS_NET_PRD_SAL_AMT': 'AVG_SAL_AMT'})
cust_sal_avg = cust_sal_avg.groupby(['INCS_NO']).AVG_SAL_AMT.mean().reset_index()

cust_sal_avg.head(2)

,INCS_NO,AVG_SAL_AMT
0,00002708cacde8c67a97934afd9b5d592acde603b6d88c...,6500.0
1,000036462ea06af85e9963ba436fccb70465ff7eccf241...,17700.0


In [52]:
cust = pd.merge(cust, cust_sal_avg, how='left', on='INCS_NO')
print(cust.shape)
cust.head(2)

(12286714, 4)


,INCS_NO,AGE,SEX_CD,AVG_SAL_AMT
0,f68d5eb51cad02327221e426d7a8878d26c13500b282bd...,60,F,NaN
1,30586a6e4e0bb50f62e007dcb09eee969afceb91486fe8...,33,F,NaN


In [57]:
cust[cust['AVG_SAL_AMT'].notna()].shape

(840984, 4)

### 3. 데이터 결합

In [54]:
temp = pd.merge(df, cust, how='left', on='INCS_NO')
print(temp.shape)
temp.head(2)

(82294, 13)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,AGE,SEX_CD,AVG_SAL_AMT
0,0005ad80698368c16258c365c05c76a21732151843,7,1,0,5,7,edda90412258df2570f4e54f2ad5b79b611b30438594bb...,2024-11-21 10:19:01.273,50.909091,0,25.0,F,NaN
1,000A204228DB4108A69885BF2BCA220E1730848747,70,13,1,23,70,51dea90cd3bb8eb26f2580e7891ebb2d38e1d820b8b85a...,2024-11-06 08:39:48.598,33.965517,1,34.0,F,18747.0


In [55]:
# cust가 누락된 데이터 수 확인
# 누락된 데이터가 너무 많다면 cust 데이터 재검토 필요

print(temp[temp['AGE'].isna()].shape)
print(temp[temp['AVG_SAL_AMT'].isna()].shape)

(71, 13)
(5137, 13)


In [74]:
temp = temp[temp['AGE'].notna()]
cv_df = temp[temp['AVG_SAL_AMT'].notna()]
print(cv_df.shape)
cv_df.head(2)

(77157, 13)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,AGE,SEX_CD,AVG_SAL_AMT
1,000A204228DB4108A69885BF2BCA220E1730848747,70,13,1,23,70,51dea90cd3bb8eb26f2580e7891ebb2d38e1d820b8b85a...,2024-11-06 08:39:48.598,33.965517,1,34.0,F,18747.0
2,000A204228DB4108A69885BF2BCA220E1730887540,20,3,0,9,20,51dea90cd3bb8eb26f2580e7891ebb2d38e1d820b8b85a...,2024-11-06 19:08:59.950,10.000000,1,34.0,F,18747.0


In [75]:
def age_coding(age):
    if age < 30:
        return 1020
    elif age < 40:
        return 30
    elif age < 50:
        return 40
    elif age < 60:
        return 50
    else:
        return 60

def sex_coding(sex):
    if sex == 'F':
        return 1
    else:
        return 0

In [82]:
# 나이 원핫인코딩
dummy_df = cv_df.copy()
dummy_df['AGE'] = dummy_df['AGE'].apply(age_coding)
dummy_df['SEX_CD'] = dummy_df['SEX_CD'].apply(sex_coding)
dummy_df = pd.get_dummies(dummy_df, columns=['AGE'], dtype=int)

# 40대 데이터 제거
dummy_df = dummy_df.drop(columns=['AGE_40'])
print(dummy_df.shape)
dummy_df.head(2)

(77157, 16)


,SESS_ID,SRCH_EFRT,PRDV_CNT,CAT_CNT,SAMGE_PAGE_CNT,EVNT_CNT,INCS_NO,LST_SESS_TIME,AVG_DEPTH,day_off,SEX_CD,AVG_SAL_AMT,AGE_30,AGE_50,AGE_60,AGE_1020
1,000A204228DB4108A69885BF2BCA220E1730848747,70,13,1,23,70,51dea90cd3bb8eb26f2580e7891ebb2d38e1d820b8b85a...,2024-11-06 08:39:48.598,33.965517,1,1,18747.0,1,0,0,0
2,000A204228DB4108A69885BF2BCA220E1730887540,20,3,0,9,20,51dea90cd3bb8eb26f2580e7891ebb2d38e1d820b8b85a...,2024-11-06 19:08:59.950,10.000000,1,1,18747.0,1,0,0,0


In [83]:
dummy_df.to_csv('./data/cv_add.csv')